# Notebook 01: Data Collection and Exploratory Analysis

**Project:** GNN-Based Battery Voltage Predictor  
**Dataset:** Materials Project Li Insertion Electrode Database  

---

## Overview

This notebook queries the Materials Project battery database for Li insertion electrode entries
using the `mp-api` Python client. We target approximately 5,000 or more entries with:

- Known average intercalation voltage (V vs Li/Li+)
- Paired charged and discharged crystal structures (for graph featurization)
- Gravimetric and volumetric specific capacity (mAh/g, mAh/cm3)

After collection, we perform exploratory data analysis (EDA) covering:

1. Voltage distribution across the full dataset
2. Chemistry family breakdown (oxides, phosphates, sulfides, etc.)
3. Capacity vs voltage scatter
4. Voltage by chemistry family (violin plot)

The processed dataset is saved to `data/battery_dataset.json` and stratified train/val/test
splits (80/10/10) are saved to `data/splits.pkl`.

In [ ]:
import os
import sys
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Add project root to path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import query_li_battery_data, load_dataset, split_dataset
from src.utils import get_chemistry_family
from src.evaluate import PALETTE

# Matplotlib publication style
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

DATA_DIR = project_root / 'data'
RESULTS_DIR = project_root / 'results'
DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

print('Directories ready.')
print(f'Project root: {project_root}')

## 1. Materials Project API Query

We use the `mp-api` client to search the insertion electrode database.
A free API key is required (register at https://materialsproject.org/api).

Set the key as an environment variable before running:
```bash
export MP_API_KEY="your_key_here"
```

The query targets:
- Working ion: Li (lithium)
- Voltage range: 0 to 6 V vs Li/Li+ (physically meaningful range)
- Fields: battery_id, average_voltage, capacity_grav, capacity_vol, framework_formula, adj_pairs

In [ ]:
API_KEY = os.environ.get('MP_API_KEY', '')
if not API_KEY:
    raise EnvironmentError(
        'MP_API_KEY environment variable not set.\n'
        'Export it before running: export MP_API_KEY="your_key"'
    )
print(f'API key loaded: {API_KEY[:8]}...')

In [ ]:
DATASET_PATH = DATA_DIR / 'battery_dataset.json'

if DATASET_PATH.exists():
    print(f'Loading cached dataset from {DATASET_PATH}')
    entries = load_dataset(str(DATASET_PATH))
else:
    print('Querying Materials Project battery database...')
    entries = query_li_battery_data(
        api_key=API_KEY,
        save_path=str(DATASET_PATH),
        max_entries=10000,
    )

print(f'Total valid entries: {len(entries)}')

## 2. Exploratory Data Analysis

We convert the raw entries to a pandas DataFrame for convenient aggregation
and then generate four publication-quality figures:

1. Voltage distribution histogram
2. Chemistry family bar chart
3. Capacity vs voltage scatter
4. Violin plot of voltage by chemistry family

In [ ]:
df = pd.DataFrame([{
    'battery_id': e['battery_id'],
    'formula': e['formula'],
    'average_voltage': e['average_voltage'],
    'capacity_grav': e['capacity_grav'],
    'capacity_vol': e['capacity_vol'],
    'chemistry_family': e['chemistry_family'],
    'num_steps': e['num_steps'],
    'has_charged': e['charged_structure'] is not None,
    'has_discharged': e['discharged_structure'] is not None,
} for e in entries])

print('Dataset shape:', df.shape)
print('\nColumn summary:')
df.describe()

In [ ]:
print('Chemistry family counts:')
print(df['chemistry_family'].value_counts().to_string())
print(f'\nVoltage range: {df["average_voltage"].min():.2f} to {df["average_voltage"].max():.2f} V')
print(f'Mean voltage: {df["average_voltage"].mean():.2f} V +/- {df["average_voltage"].std():.2f} V')
print(f'Median voltage: {df["average_voltage"].median():.2f} V')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(df['average_voltage'], bins=60, color='#2196F3', alpha=0.80, edgecolor='none')
ax.axvline(df['average_voltage'].mean(), color='#F44336', linewidth=1.5,
           linestyle='--', label=f'Mean: {df["average_voltage"].mean():.2f} V')
ax.axvline(df['average_voltage'].median(), color='#FF9800', linewidth=1.5,
           linestyle=':', label=f'Median: {df["average_voltage"].median():.2f} V')

ax.set_xlabel('Average Voltage (V vs Li/Li+)')
ax.set_ylabel('Number of Electrode Entries')
ax.set_title('Distribution of Li Intercalation Voltages (Materials Project)')
ax.legend(frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig01_voltage_distribution.png')
plt.show()
print('Saved: fig01_voltage_distribution.png')

In [ ]:
chem_counts = df['chemistry_family'].value_counts().sort_values(ascending=True)
chem_colors = [PALETTE.get(f, '#607D8B') for f in chem_counts.index]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(range(len(chem_counts)), chem_counts.values,
        color=chem_colors, alpha=0.85, edgecolor='none')
ax.set_yticks(range(len(chem_counts)))
ax.set_yticklabels(chem_counts.index)
ax.set_xlabel('Number of Electrode Entries')
ax.set_title('Electrode Count by Chemistry Family')

for i, n in enumerate(chem_counts.values):
    pct = n / len(df) * 100
    ax.text(n + 5, i, f'{n} ({pct:.1f}%)', va='center', fontsize=9)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig01_chemistry_breakdown.png')
plt.show()
print('Saved: fig01_chemistry_breakdown.png')

In [ ]:
df_plot = df[(df['capacity_grav'] > 0) & (df['capacity_grav'] < 1000)].copy()

fig, ax = plt.subplots(figsize=(7, 5))

for fam, grp in df_plot.groupby('chemistry_family'):
    ax.scatter(
        grp['capacity_grav'], grp['average_voltage'],
        s=15, alpha=0.5, label=fam,
        color=PALETTE.get(fam, '#607D8B'), linewidths=0
    )

ax.set_xlabel('Gravimetric Specific Capacity (mAh/g)')
ax.set_ylabel('Average Voltage (V vs Li/Li+)')
ax.set_title('Specific Capacity vs Average Voltage')
ax.legend(title='Chemistry', loc='upper right',
          frameon=True, framealpha=0.8, fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add energy density contours (E = V * Q, in Wh/kg)
for energy in [200, 400, 600, 800]:
    q_vals = np.linspace(50, 900, 100)
    v_vals = energy / q_vals
    mask = (v_vals > 0.5) & (v_vals < 6)
    ax.plot(q_vals[mask], v_vals[mask], 'k--', linewidth=0.7, alpha=0.3)
    idx = len(q_vals) // 2
    ax.text(q_vals[idx], v_vals[idx] + 0.05,
            f'{energy} Wh/kg', fontsize=7.5, color='gray', ha='center')

plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig01_capacity_vs_voltage.png')
plt.show()
print('Saved: fig01_capacity_vs_voltage.png')

In [ ]:
families_ordered = df.groupby('chemistry_family')['average_voltage'].median().sort_values(ascending=False).index.tolist()
df_violin = df[df['chemistry_family'].isin(families_ordered)].copy()
df_violin['chemistry_family'] = pd.Categorical(df_violin['chemistry_family'], categories=families_ordered, ordered=True)

fig, ax = plt.subplots(figsize=(8, 4))

for i, fam in enumerate(families_ordered):
    subset = df_violin[df_violin['chemistry_family'] == fam]['average_voltage'].values
    if len(subset) < 2:
        continue
    # Violin patch
    parts = ax.violinplot(subset, positions=[i], showmedians=True, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(PALETTE.get(fam, '#607D8B'))
        pc.set_alpha(0.75)
    parts['cmedians'].set_color('#111111')
    parts['cmedians'].set_linewidth(1.5)

ax.set_xticks(range(len(families_ordered)))
ax.set_xticklabels(families_ordered, rotation=25, ha='right')
ax.set_ylabel('Average Voltage (V vs Li/Li+)')
ax.set_title('Voltage Distribution by Chemistry Family')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig01_voltage_by_chemistry.png')
plt.show()
print('Saved: fig01_voltage_by_chemistry.png')

## 3. Train / Validation / Test Split

We split the dataset 80/10/10 using stratified sampling by chemistry family.
Stratification ensures that each split contains a representative proportion
of every chemistry class, preventing data leakage from over-representation
of any single family in the training set.

Split sizes and per-family breakdown are printed below.

In [ ]:
train_entries, val_entries, test_entries = split_dataset(
    entries, train_frac=0.80, val_frac=0.10, seed=42
)

print(f'Train: {len(train_entries):4d} entries')
print(f'Val:   {len(val_entries):4d} entries')
print(f'Test:  {len(test_entries):4d} entries')
print(f'Total: {len(train_entries)+len(val_entries)+len(test_entries):4d} entries')

# Per-family breakdown
for split_name, split_data in [('Train', train_entries), ('Val', val_entries), ('Test', test_entries)]:
    fam_counts = pd.Series([e['chemistry_family'] for e in split_data]).value_counts()
    print(f'\n{split_name} family distribution:')
    print(fam_counts.to_string())

In [ ]:
splits = {'train': train_entries, 'val': val_entries, 'test': test_entries}
with open(DATA_DIR / 'splits.pkl', 'wb') as f:
    pickle.dump(splits, f)

print(f'Saved splits to {DATA_DIR / "splits.pkl"}')
print('Notebook 01 complete. Proceed to 02_feature_engineering.ipynb')